# 04 · Validación del dataset

Qué hacen los labs antes de entrenar, aplicado a nuestros 65 episodios de
`pick_mameluco`. La investigación de fondo, con fuentes, está en
`docs/08_validacion_de_datos.md`; acá van los chequeos y los gráficos.

Dos ideas que ordenan todo:

1. **No hay un notebook estándar.** Lo que se repite entre RT-1, π0.5, DROID,
   GR00T, TRI y compañía son reglas sueltas: entrenar sólo con éxitos, recortar
   los extremos quietos, buscar saltos y desincronización, y **mirar los
   episodios a mano**. Los números marcan candidatos; la decisión es visual.
2. **Los umbrales se declaran antes de mirar.** Están en
   `toolkit/validation/criteria.py`. Si alguno se cambia después de ver los
   datos, se anota acá por qué.

Cada sección termina sumando columnas a una tabla con una fila por episodio
(`flags`). Al final esa tabla dice qué revisar.

Un dato lindo para tener en la cabeza: alguien ya descartó a mano 4 de estos
65 episodios para entrenar la primera ACT (notebook 03). Vamos a ver si los
chequeos los encuentran solos.

In [ ]:
import torch  # noqa: F401  (antes que lerobot, ver notebook 01)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.validation import coherence, coverage, demonstrations, drift, integrity, signals, visual
from toolkit.validation.criteria import Criteria
from toolkit.validation.episodes import joint_groups, load_episodes, read_episode_meta
from toolkit.validation.robust import robust_z

plt.rcParams["figure.dpi"] = 110
ROOT = DATASETS_DIR / "local" / "pick_mameluco"   # los 65 episodios, sin filtrar
CAMERA = "observation.images.cam_right_high"
FPS = 30
criteria = Criteria()
criteria

## 0. Carga y trazabilidad

El conversor renumera los episodios 0..64. Para poder hablar de "el orden de
grabación" o de "el episodio 0008" guardamos el nombre de la carpeta cruda de
cada uno (`meta/source_episodes.json`).

In [ ]:
episodes = load_episodes(ROOT)
groups = joint_groups(ROOT)
NAMES = groups.names
flags = pd.DataFrame({"episodio": [e.source for e in episodes], "cuadros": [e.length for e in episodes]},
                     index=[e.index for e in episodes])
print(len(episodes), "episodios,", flags["cuadros"].sum(), "cuadros")
print("brazos:", len(groups.arms), "articulaciones · manos:", len(groups.hands))
flags.head()

## 1. Inventario y esquema

Lo más barato y lo primero: ¿los conteos cuadran? ¿hay NaN? ¿están las
estadísticas que necesita la normalización?

In [ ]:
display(integrity.schema_report(ROOT))
print("valores no finitos:", int(integrity.invalid_values(episodes).values.sum()))

## 2. Integridad temporal y de video

**El chequeo clásico** es mirar que el tiempo entre cuadros sea siempre 1/30 s.
Si falta un cuadro, aparece un intervalo de 2/30.

In [ ]:
steps = integrity.timestamp_steps(ROOT)
print("intervalos distintos (s):", np.unique(np.round(steps, 5)))

Siempre 1/30 exacto. **Y eso no prueba nada**: nuestras grabaciones no guardan
la hora de cada cuadro, sólo un número de orden, y el conversor inventa el
tiempo como `frame_index / 30`. Ningún chequeo de timestamps puede encontrar
cuadros perdidos en este dataset. (Para las próximas grabaciones conviene
guardar la hora real de cada cuadro; GR00T lo pide explícitamente.)

El sustituto indirecto: si la cámara se trabó, la imagen se repite mientras el
robot se mueve. Decodificamos todo el video una vez y medimos cuánto cambia
cada cuadro respecto del anterior (tarda unos 20 s).

In [ ]:
frames = visual.assign_episodes(visual.frame_metrics(ROOT, CAMERA), ROOT, CAMERA)
meta = read_episode_meta(ROOT)
print("cuadros de video:", len(frames), "· coinciden con el parquet en cada episodio:",
      bool((frames.groupby("episode_index").size().to_numpy() == meta["length"].to_numpy()).all()))

suspicious = []
for e in episodes:
    change = frames.loc[frames.episode_index == e.index, "change"].to_numpy()
    moving = np.concatenate([[0], np.abs(np.diff(e.state[:, groups.arms], axis=0)).max(axis=1)])
    for start, length in visual.frozen_runs(change, criteria.frozen_frame_diff):
        if moving[start:start + length].max() > criteria.pause_velocity_rad:
            suspicious.append((e.source, start, length))
flags["video_trabado"] = flags["episodio"].isin({s for s, _, _ in suspicious})
print("tramos con imagen quieta y brazos en movimiento:", suspicious or "ninguno")

## 3. Señales articulares

### Saltos

Un salto es un cambio entre dos cuadros que el motor no puede hacer:
típicamente un cuadro perdido, un glitch de la cinemática inversa o un reinicio
de la teleoperación. Sin los límites de velocidad del robot a mano, el umbral
sale de los datos: más del doble del percentil 99,9 del paso de esa
articulación.

(Primera versión usaba mediana + 8·MAD como sugiere la investigación. No sirve
acá: la mayoría de los cuadros están quietos, la mediana del paso es ~0 y
cualquier movimiento parecía un salto. Cambio anotado en `criteria.py`.)

In [ ]:
jumps_state = signals.jump_counts(episodes, "state", criteria)
jumps_action = signals.jump_counts(episodes, "action", criteria)
flags["saltos_estado"] = jumps_state.sum(axis=1)
flags["saltos_accion"] = jumps_action.sum(axis=1)

ratio = signals.max_step_ratio(episodes, "state", criteria)
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(ratio.T, aspect="auto", cmap="magma", vmin=0, vmax=1.5)
ax.set_yticks(range(len(NAMES))); ax.set_yticklabels(NAMES, fontsize=6)
ax.set_xticks(range(0, len(episodes), 4)); ax.set_xticklabels(flags["episodio"].iloc[::4], rotation=90, fontsize=6)
ax.set_title("paso más grande / umbral de salto (estado) · arriba de 1 es salto")
plt.colorbar(im); plt.tight_layout()
flags.loc[flags.saltos_estado + flags.saltos_accion > 0, ["episodio", "saltos_estado", "saltos_accion"]]

Las señales son suaves: casi ningún paso pasa el umbral. El que más acumula
saltos en el estado es `episode_0008`. Anotado.

### Brusquedad

Puntaje del visualizador oficial de LeRobot: el cambio medio de la acción por
cuadro, relativo al rango de cada articulación. Un episodio muy por encima del
resto suele ser un operador apurado o un seguimiento ruidoso.

In [ ]:
jerk = signals.jerkiness(episodes)
flags["brusquedad_z"] = robust_z(jerk).round(1)
ax = pd.Series(jerk, index=flags["episodio"]).plot.bar(figsize=(14, 3), color=["tab:red" if z > criteria.robust_z else "tab:blue" for z in flags["brusquedad_z"]])
ax.set_xticks(range(0, len(episodes), 4)); ax.set_xticklabels(flags["episodio"].iloc[::4], rotation=90, fontsize=7)
ax.set_title("brusquedad por episodio"); plt.tight_layout()

### Saturación de las manos y articulaciones congeladas

Las manos van de 0 (abierta) a 1 (cerrada al máximo). Estar en 0 es la mano en
reposo, no es problema. Estar clavado en 1 sí: la orden pide más de lo que la
mano puede dar.

Una articulación casi congelada es un riesgo silencioso: con normalización por
media y desvío, su ruido se convierte en muchos desvíos.

In [ ]:
saturation = signals.hand_saturation(episodes, groups.hands, "action", criteria)
print("fracción del tiempo con la orden de la mano en el tope (promedio por articulación):")
display(pd.Series(saturation.mean(axis=0), index=[NAMES[i] for i in groups.hands]).round(3).to_frame("en el tope"))
flags["mano_en_tope"] = saturation.max(axis=1).round(2)

ranges = signals.joint_ranges(episodes)
ax = pd.Series(ranges, index=NAMES).plot.bar(figsize=(12, 3)); ax.set_title("rango q01–q99 de la acción por articulación"); plt.tight_layout()
print("congeladas (brazos):", [NAMES[i] for i in signals.frozen_joints(episodes, criteria, groups.arms)])
print("congeladas (manos):", [NAMES[i] for i in signals.frozen_joints(episodes, criteria, groups.hands)])

## 4. Acción contra estado

### Lag de control

La acción es el objetivo; el motor llega unos cuadros después. Buscamos el
corrimiento que mejor alinea a ambos.

In [ ]:
lags = coherence.lag_matrix(episodes, list(range(len(NAMES))))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
example = episodes[10]
for j in [NAMES.index("kRightElbow"), NAMES.index("kRightHandIndex")]:
    axes[0].plot(coherence.lag_curve(example, j), label=NAMES[j])
axes[0].set_xlabel("corrimiento (cuadros)"); axes[0].set_ylabel("|acción − estado futuro|"); axes[0].legend(); axes[0].set_title(example.source)
axes[1].boxplot([lags[:, groups.arms].ravel(), lags[:, groups.hands].ravel()], labels=["brazos", "manos"])
axes[1].set_ylabel("lag (cuadros)"); plt.tight_layout()
print("lag mediano: brazos", np.median(lags[:, groups.arms]), "cuadros · manos", np.median(lags[:, groups.hands]), "cuadros")
flags["lag_brazos"] = np.median(lags[:, groups.arms], axis=1)

Brazos: ~3 cuadros (100 ms), retraso normal de control y muy chico al lado de
un bloque de 48. Manos: bastante más lento. Un episodio con lag muy distinto al
resto sería síntoma de desincronización en la grabación.

### Error de seguimiento en las manos

Lo que la mano no alcanza. Hipótesis de la investigación: al apretar la tela,
la mano no llega a la posición pedida.

In [ ]:
j = NAMES.index("kRightHandIndex")
lag = int(np.median(lags[:, j]))
error = coherence.tracking_error(example, lag)[:, j]
t = np.arange(len(error)) / FPS
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, example.action[: len(error), j], label="orden")
ax.plot(t, example.state[lag:, j], label=f"estado (corrido {lag} cuadros)")
ax.fill_between(t, 0, error, alpha=0.3, label="lo que no alcanzó")
ax.set_title(f"{example.source} · {NAMES[j]}"); ax.legend(); plt.tight_layout()
print("¿la acción es copia del estado en algún episodio?", any(coherence.action_copies_state(e) for e in episodes))

### Autocorrelación y largo del bloque

¿Cada cuánto la acción "se vuelve otra"? El visualizador de LeRobot sugiere
leerlo como un largo natural de bloque. Es una heurística, no un resultado de
paper, pero es interesante compararla con el 48 que usa nuestra ACT.

In [ ]:
print("la autocorrelación de los brazos cae debajo de 0,5 a los",
      coherence.decorrelation_lag(episodes, groups.arms), "cuadros (mediana)")

## 5. Calidad de cada demostración

### Duración y extremos quietos

Todos los labs recortan los tramos del principio (el operador todavía no
arrancó) y del final (ya terminó y sigue grabando). TRI usa "moverse más de
5 cm de la pose inicial"; acá, la versión articular: más de 0,05 rad.

In [ ]:
ends = [demonstrations.idle_ends(e, groups.arms, criteria) for e in episodes]
flags["quieto_inicio_s"] = [round(x.head / FPS, 1) for x in ends]
flags["quieto_final_s"] = [round(x.tail / FPS, 1) for x in ends]
flags["duracion_z"] = robust_z(flags["cuadros"].to_numpy()).round(1)

active = flags["cuadros"] / FPS - flags["quieto_inicio_s"] - flags["quieto_final_s"]
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.bar(flags["episodio"], flags["quieto_inicio_s"], color="0.7", label="quieto al inicio")
ax.bar(flags["episodio"], active.clip(lower=0), bottom=flags["quieto_inicio_s"], label="activo")
ax.bar(flags["episodio"], flags["quieto_final_s"], bottom=flags["quieto_inicio_s"] + active.clip(lower=0), color="0.4", label="quieto al final")
ax.set_xticks(range(0, len(episodes), 4)); ax.set_xticklabels(flags["episodio"].iloc[::4], rotation=90, fontsize=7)
ax.set_ylabel("segundos"); ax.legend(); plt.tight_layout()
print(f"en total hay {(flags.quieto_inicio_s + flags.quieto_final_s).sum():.0f} s recortables de {flags.cuadros.sum() / FPS:.0f} s")

Ojo antes de recortar: TRI encontró que recortar ayudó a sus políticas de una
tarea pero empeoró su modelo grande preentrenado. Si lo hacemos, es un
experimento con métrica declarada, no una limpieza "obvia".

### Pausas a mitad de camino

In [ ]:
flags["pausas"] = [round(demonstrations.mid_episode_pauses(e, groups.arms, criteria), 2) for e in episodes]
print("episodios con alguna pausa de ≥ 0,5 s en la parte activa:", int((flags.pausas > 0).sum()))

### ¿Hubo agarre?

El chequeo más importante de todos: los labs entrenan **sólo con éxitos**. No
tenemos etiqueta de éxito, pero tenemos un indicador fuerte: si la mano nunca
se cerró, no agarró nada.

In [ ]:
closure_left = [demonstrations.hand_closure(e, groups.left_hand) for e in episodes]
closure_right = [demonstrations.hand_closure(e, groups.right_hand) for e in episodes]
flags["cierre_max"] = [round(max(l.max(), r.max()), 2) for l, r in zip(closure_left, closure_right)]
flags["sin_agarre"] = flags["cierre_max"] < criteria.grasp_min_closure
flags["ciclos_agarre"] = [max(demonstrations.grasp_cycles(l, 0.3, 0.15), demonstrations.grasp_cycles(r, 0.3, 0.15))
                          for l, r in zip(closure_left, closure_right)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(flags["cuadros"] / FPS, flags["cierre_max"], c=np.where(flags["sin_agarre"], "tab:red", "tab:blue"))
for _, row in flags[flags.sin_agarre].iterrows():
    ax.annotate(row["episodio"], (row["cuadros"] / FPS, row["cierre_max"]), fontsize=7)
ax.axhline(criteria.grasp_min_closure, ls="--", color="0.5")
ax.set_xlabel("duración (s)"); ax.set_ylabel("cierre máximo de la mano"); plt.tight_layout()
flags.loc[flags.sin_agarre, ["episodio", "cuadros", "cierre_max"]]

Tres episodios cortísimos donde la mano nunca cierra: intentos que se cortaron.
Son tres de los cuatro que se descartaron a mano.

### Consistencia entre demostraciones

El ruido del operador perjudica (Belkhale et al.): conviene que todas las
demostraciones hagan lo mismo de la misma forma. Superponemos las trayectorias
estiradas a la misma duración.

In [ ]:
speeds = np.array([demonstrations.mean_speed(e, groups.arms) for e in episodes])
ok = ~flags["sin_agarre"].to_numpy()
cv = speeds[ok].std() / speeds[ok].mean()
verdict = "consistente" if cv < criteria.speed_cv_consistent else "variable" if cv < criteria.speed_cv_large else "muy variable"
print(f"coeficiente de variación de la velocidad (sin los fallidos): {cv:.2f} → {verdict}")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, name in zip(axes, ["kRightShoulderPitch", "kRightElbow", "kLeftElbow"]):
    j = NAMES.index(name)
    curves = np.stack([demonstrations.time_normalized(e.action[:, j]) for e, keep in zip(episodes, ok) if keep])
    ax.plot(curves.T, color="tab:blue", alpha=0.15)
    ax.plot(np.median(curves, axis=0), color="black", lw=2)
    ax.set_title(name); ax.set_xlabel("% del episodio")
plt.tight_layout()

## 6. Calidad visual

Desenfoque (varianza del Laplaciano: pocos bordes nítidos = borroso),
iluminación y exposición, por episodio. Los valores son relativos a este
dataset, no absolutos.

In [ ]:
per_episode = frames.groupby("episode_index")[["sharpness", "luminance", "overexposed"]].median()
flags["nitidez_z"] = robust_z(per_episode["sharpness"].to_numpy()).round(1)
flags["luz_z"] = robust_z(per_episode["luminance"].to_numpy()).round(1)
fig, axes = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True)
axes[0].plot(flags["episodio"], per_episode["sharpness"], "o-"); axes[0].set_ylabel("nitidez")
axes[1].plot(flags["episodio"], per_episode["luminance"], "o-", color="tab:orange"); axes[1].set_ylabel("luminancia")
axes[1].set_xticks(range(0, len(episodes), 4)); axes[1].set_xticklabels(flags["episodio"].iloc[::4], rotation=90, fontsize=7)
plt.tight_layout()

Y los cuadros más borrosos del dataset, para ver si "borroso" significa algo:

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("local/pick_mameluco", root=ROOT, video_backend="pyav")
starts = meta.set_index("episode_index")["dataset_from_index"]
blurriest = frames.nsmallest(4, "sharpness")
fig, axes = plt.subplots(1, 4, figsize=(16, 2.6))
for ax, (_, row) in zip(axes, blurriest.iterrows()):
    ax.imshow(ds[int(starts[row.episode_index] + row.frame_index)][CAMERA].permute(1, 2, 0))
    ax.set_title(f"{flags.loc[row.episode_index, 'episodio']} · cuadro {int(row.frame_index)}", fontsize=8); ax.axis("off")
plt.tight_layout()

## 7. Deriva a lo largo de la grabación

¿Algo cambió a mitad de la campaña? Nosotros sabemos que sí: hubo un arreglo
del pulgar. Veamos si los datos lo muestran sin decirles dónde buscar.
Graficamos, para cada episodio en orden de grabación, el máximo que alcanzó
cada articulación de la mano izquierda, y buscamos el mejor punto de corte.

In [ ]:
left = groups.left_hand
maxima = np.stack([e.state[:, left].max(axis=0) for e in episodes])
fig, ax = plt.subplots(figsize=(14, 3.5))
for k, j in enumerate(left):
    ax.plot(flags["episodio"], maxima[:, k], "o-", ms=3, label=NAMES[j])
ax.set_xticks(range(0, len(episodes), 4)); ax.set_xticklabels(flags["episodio"].iloc[::4], rotation=90, fontsize=7)
ax.set_ylabel("máximo del estado"); ax.legend(fontsize=7, ncol=3); plt.tight_layout()

for k, j in enumerate(left):
    cp = drift.change_point(maxima[:, k])
    print(f"{NAMES[j]:18s} corte después de {flags['episodio'].iloc[cp.split - 1]} · puntaje {cp.score:.1f}")

En `episode_0008` el pulgar (`ThumbAux`) llega a 0,90 y el índice a 0,56; en
todos los episodios siguientes el pulgar no pasa de 0,57 y el índice ronda
0,07. Para esas dos articulaciones el mejor corte cae justo después del 0008,
con los puntajes más altos. Es el arreglo del pulgar (y su efecto en el índice),
encontrado por los datos sin decirles dónde buscar. Ese es el cuarto episodio
que se descartó a mano.

(Los tres pozos a cero son los intentos sin agarre de la sección 5.)

In [ ]:
thumb = NAMES.index("kLeftHandThumbAux")
cp = drift.change_point(np.array([e.state[:, thumb].max() for e in episodes]))
flags["regimen_pulgar"] = np.where(np.arange(len(episodes)) < cp.split, "antes", "después")

## 8. Diversidad y cobertura

¿Las demostraciones arrancan siempre igual? Si la pose inicial es siempre la
misma, la política no aprende a arrancar desde otro lado. PCA comprime las 26
articulaciones a 2 ejes para poder mirarlas.

In [ ]:
initial = np.stack([e.state[0] for e in episodes])
projection, explained = coverage.pca(initial)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sc = axes[0].scatter(projection[:, 0], projection[:, 1], c=np.arange(len(episodes)), cmap="viridis")
axes[0].set_title(f"pose inicial (PCA, {explained.sum():.0%} de la varianza)"); plt.colorbar(sc, ax=axes[0], label="orden de grabación")

firsts = [0, 20, 40, 60]
axes[1].axis("off")
grid = np.concatenate([ds[int(starts[i])][CAMERA].permute(1, 2, 0).numpy() for i in firsts], axis=1)
axes[1].imshow(grid); axes[1].set_title("primer cuadro de 4 episodios")
plt.tight_layout()

## 9. Split y fuga de información

Ahora el dataset con el que se entrenó ACT (61 episodios) y su split.

**Regla 1:** se separa por episodio, nunca por cuadro (cuadros vecinos son casi
idénticos).

In [ ]:
from toolkit.checks.normalization import compare, load_policy_stats
from toolkit.dataset.stats import subset_stats

ACT_ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
train_ids, heldout_ids = set(split["train_episode_indices"]), set(split["heldout_episode_indices"])
print("¿algún episodio en los dos lados?", bool(train_ids & heldout_ids))

**Regla 2: la normalización tampoco puede ver el held-out.** La media y el
desvío con que se normaliza son información del dataset. Si se calculan con
los episodios de evaluación incluidos, un poquito de ellos se filtra al
entrenamiento.

In [ ]:
policy_stats = load_policy_stats(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model")
all_61 = subset_stats(ACT_ROOT)
train_only = subset_stats(ACT_ROOT, exclude=heldout_ids)
for label, stats in [("con los 61", all_61), ("sólo train (54)", train_only)]:
    for gap in compare(stats, policy_stats):
        print(f"{label:16s} {gap.feature:18s} diferencia con el modelo: {gap.max_mean_shift:.1e} desvíos")

La normalización del modelo coincide con la de **los 61**: el held-out entró.
Es una fuga chica (la diferencia con "sólo train" es de centésimas de desvío),
pero existe; al reentrenar, la normalización se calcula sólo con train.

**Regla 3: ¿el held-out se parece al train?** Para cada cuadro de evaluación,
la distancia al cuadro más parecido del train (en el espacio de las 26
articulaciones). Si es tan chica como la distancia típica dentro del train,
la evaluación es "en distribución": mide si aprendió, no si generaliza.

In [ ]:
act_episodes = load_episodes(ACT_ROOT)
def stack(ids, step=3):
    return np.concatenate([e.state[::step] for e in act_episodes if e.index in ids])

train_states, heldout_states = stack(train_ids), stack(heldout_ids)
scale = train_states.std(axis=0) + 1e-9
heldout_to_train = coverage.nearest_distances(heldout_states / scale, train_states / scale)

# referencia: cada episodio de train contra el resto del train
within = np.concatenate([
    coverage.nearest_distances(e.state[::3] / scale, stack(train_ids - {e.index}) / scale)
    for e in act_episodes if e.index in list(train_ids)[:10]
])
for values, label in [(within, "train contra train (otro episodio)"), (heldout_to_train, "held-out contra train")]:
    plt.plot(np.sort(values), np.linspace(0, 1, len(values)), label=label)
plt.xlabel("distancia al vecino más cercano (desvíos)"); plt.ylabel("fracción de cuadros"); plt.legend(); plt.tight_layout()

Si las dos curvas se pisan, el held-out es "más de lo mismo". Con un split
intercalado (0, 10, 20, …) es lo esperable: sirve para ver si aprendió, no para
ver si aguanta cambios. Un split complementario sería **por bloque temporal**
(los últimos episodios grabados), que es lo que hace `eval_split` en LeRobot.

## 10. Curación automática (para más adelante)

Con ~60 demostraciones el método más práctico es **DemInf** (no necesita
rollouts). Viene después de filtrar fracasos, porque premia lo "predecible" y
tiende a valorar las pausas. Queda para cuando reentrenemos.

## 11. Resumen y decisión

In [ ]:
flags["a_revisar"] = (
    flags["sin_agarre"]
    | (flags["regimen_pulgar"] == "antes")
    | (flags["brusquedad_z"].abs() > criteria.robust_z)
    | (flags["duracion_z"].abs() > criteria.robust_z)
    | (flags["saltos_estado"] > 1)
    | flags["video_trabado"]
    | (flags["nitidez_z"].abs() > criteria.robust_z)
)
review = flags[flags.a_revisar]
review[["episodio", "cuadros", "cierre_max", "regimen_pulgar", "brusquedad_z", "duracion_z", "saltos_estado", "nitidez_z"]]

¿Coincide con lo que se descartó a mano para la primera ACT?

In [ ]:
excluded_by_hand = {"episode_0008", "episode_0015", "episode_0028", "episode_0058"}
found = set(review["episodio"])
print("descartados a mano:", sorted(excluded_by_hand))
print("marcados por los chequeos:", sorted(found))
print("coinciden:", sorted(found & excluded_by_hand), "· sólo chequeos:", sorted(found - excluded_by_hand), "· sólo a mano:", sorted(excluded_by_hand - found))

**La decisión es visual.** Antes de sacar un episodio, se mira: primero, medio
y último cuadro de cada marcado.

Lo que se ve en nuestra corrida:

- `episode_0008`: otra escena (una caja debajo de la prenda, otro encuadre) y
  el pulgar de antes del arreglo. Fuera.
- `episode_0015`, `0028`, `0058`: las manos quedan al costado de la prenda y no
  agarran. Fuera.
- `episode_0055`: en el último cuadro la prenda está levantada frente a la
  cámara. **Salió bien**, sólo fue más rápido que el resto. Falso positivo: se
  queda. Es exactamente por esto que la decisión no la toma la tabla.

In [ ]:
fig, axes = plt.subplots(len(review), 3, figsize=(12, 2.4 * len(review)), squeeze=False)
for row_axes, (idx, row) in zip(axes, review.iterrows()):
    for ax, k in zip(row_axes, [0, row.cuadros // 2, row.cuadros - 1]):
        ax.imshow(ds[int(starts[idx] + k)][CAMERA].permute(1, 2, 0)); ax.axis("off")
    row_axes[0].set_title(row["episodio"], loc="left", fontsize=9)
plt.tight_layout()

Lo que queda para la próxima vuelta de datos (no de este notebook):

- Guardar la hora real de cada cuadro al grabar.
- Etiquetar éxito/fracaso al grabar (como DROID), en vez de inferirlo.
- Variar más la posición inicial de la prenda: la diversidad del entorno pesa
  más que la cantidad de demostraciones (Data Scaling Laws, RT-1).